In [1]:
import argparse, json, re
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq

/home/wenduoc/mambaforge/envs/spatialagent/lib/python3.9/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
/home/wenduoc/mambaforge/envs/spatialagent/lib/python3.9/site-packages/numba/core/decorators.py:246: RuntimeWarning: nopython is set for njit and is ignored
  warnings.warn('nopython is set for njit and is ignored', RuntimeWarning)


# Explore data

https://cells.ucsc.edu/?ds=hoc

PCW = post-conception weeks


In [2]:
ad = sc.read_h5ad("/work/magroup/shared/agent/data/Developing_HumanHeart_Farah/overall_merfish.h5ad")

In [9]:
ad

AnnData object with n_obs × n_vars = 228635 × 238
    obs: 'sample_id', 'batch', 'n_counts', 'leiden', 'zone_cluster', 'communities', 'complexity', 'populations', 'purity'
    uns: 'communities_colors', 'leiden', 'leiden_colors', 'neighbors', 'pca', 'populations_colors', 'sample_id_colors', 'umap', 'zone_cluster_colors'
    obsm: 'X_pca', 'X_umap', 'spatial'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'

adata.obs (per-cell annotations)

- sample_id / batch – which section/replicate the cell came from.
    
- n_counts – total transcripts detected per cell (sum over the 238 targets).
    
- leiden – unsupervised expression-space clusters from the 238-gene matrix (basis for UMAP).
    
- populations – cell type / major lineage labels (e.g., vCM subtypes, endothelial, EPDC, fibroblast, etc.) transferred/curated for MERFISH. 
    Nature
    
- zone_cluster – clustering of cell zones (neighborhoods) prior to community calling (fine-grained spatial groups). 
    Nature
    
- communities – the higher-level cellular community (CC) assignment for each cell’s zone; these map to anatomical structures (e.g., AV canal, valves, LV/RV layers, conduction system). 
    Nature
    
- complexity – diversity of cell types within the local zone (higher = more mixed).
    
- purity – dominance of the top cell type in the zone (higher = more uniform). In the paper, some ventricular communities (e.g., Intermediate-LV) show high complexity / low purity. 
    Nature

adata.uns (dataset-level stuff)

- *_colors – color palettes for categorical labels (stable plotting colors).

- neighbors, pca, umap – saved results so you can plot immediately without recomputing.

adata.obsm (embeddings)

- X_pca – per-cell PC scores.

- X_umap – 2D UMAP coordinates.

- spatial – per-cell XY tissue coordinates (plot this to see the heart section layout).

adata.varm / adata.obsp

- PCs – PCA loadings (genes × PCs).

- connectivities / distances – KNN graph adjacency & distances used for Leiden/UMAP.

In [4]:
ad.obs

,sample_id,batch,n_counts,leiden,zone_cluster,communities,complexity,populations,purity
6-R77_4C4,R77_4C4,R77_4C4,86.0,6,4,AVN/AV Ring,8,VIC,0.544534
8-R77_4C4,R77_4C4,R77_4C4,148.0,6,1,Valve,8,VIC,0.625984
9-R77_4C4,R77_4C4,R77_4C4,100.0,6,1,Valve,8,VIC,0.583665
10-R77_4C4,R77_4C4,R77_4C4,70.0,6,1,Valve,9,VIC,0.766393
12-R77_4C4,R77_4C4,R77_4C4,63.0,6,1,Valve,8,VIC,0.596838
...,...,...,...,...,...,...,...,...,...
6660345-R78_4C15,R78_4C15,R78_4C15,70.0,6,1,Valve,9,VIC,0.724675
6660348-R78_4C15,R78_4C15,R78_4C15,83.0,24,1,Valve,8,VEC,0.747340
6660352-R78_4C15,R78_4C15,R78_4C15,70.0,24,1,Valve,7,VEC,0.771772
6660357-R78_4C15,R78_4C15,R78_4C15,118.0,6,1,Valve,9,VIC,0.657963


In [28]:
ad.obs['leiden'].unique()

['6', '16', '17', '29', '10', ..., '1', '26', '33', '15', '28']
Length: 32
Categories (32, object): ['0', '1', '2', '3', ..., '30', '31', '32', '33']

In [27]:
ad.obs['populations'].unique(). # cell type labels

['VIC', 'vCM-LV-AV', 'vCM-RV-AV', 'ncCM-AVC-like', 'EPDC', ..., 'vCM-LV-Trabecular', 'Epicardial', 'Neuronal', 'adFibro', 'ncCM-IFT-like']
Length: 27
Categories (27, object): ['ncCM-AVC-like', 'vCM-LV-Compact', 'vCM-RV-Compact', 'EPDC', ..., 'vCM-LV-AV', 'vCM-RV-AV', 'vEndocardial', 'vFibro']

In [24]:
ad.obs['communities'].unique() # region labels

['AVN/AV Ring', 'Valve', 'Right Atria', 'VCS', 'Mus. Valve Leaf.', ..., 'Inner-LV', 'Subepicardial', 'Left Atria', 'OFT', 'IFT/SAN']
Length: 13
Categories (13, object): ['AVN/AV Ring', 'VCS', 'IFT/SAN', 'Inner-LV', ..., 'Mus. Valve Leaf.', 'Right Atria', 'Subepicardial', 'Valve']

In [25]:
ad.obs['zone_cluster'].unique()

['4', '1', '3', '10', '5', ..., '2', '11', '9', '7', '12']
Length: 13
Categories (13, object): ['0', '1', '2', '3', ..., '9', '10', '11', '12']

In [26]:
ad.obs['complexity'].unique()

array([ 8,  9, 10, 11, 12, 13, 14, 15, 16,  5,  7,  6,  4,  3,  2, 17, 18,
       19,  1, 20])

In [5]:
ad.var 

""
ABCC9
ADAMTS6
ADAMTS8
ADGRL1
ADGRL2
...
UPK3B
VCAN
VSNL1
WT1


In [10]:
ad.obsm['spatial']

array([[-8360.24178439,  -264.45033689],
       [-8315.43417175,  -251.40620106],
       [-8339.86084146,  -256.9504057 ],
       ...,
       [ 4205.7522308 ,  -240.51815826],
       [ 4317.30226271,  -217.38076107],
       [ 4252.52992873,  -227.24918918]])

In [16]:
ad.obsm['X_pca'].shape

(228635, 50)

In [19]:
ad.obsm['X_umap'].shape

(228635, 2)

In [23]:
ad.obsp['distances'].shape

(228635, 228635)

In [17]:
ad.X

array([[0.       , 0.       , 0.       , ..., 0.       , 0.       ,
        0.       ],
       [0.       , 0.       , 0.       , ..., 0.       , 0.       ,
        0.       ],
       [0.       , 0.       , 0.       , ..., 0.       , 0.       ,
        0.       ],
       ...,
       [0.       , 0.       , 0.       , ..., 0.       , 0.       ,
        0.       ],
       [0.       , 0.       , 0.       , ..., 0.       , 0.       ,
        1.9721961],
       [0.       , 1.803267 , 0.       , ..., 0.       , 0.       ,
        0.       ]], dtype=float32)

In [13]:
ad.X.shape

(228635, 238)

In [ ]:
ad = sc.read_h5ad("/work/magroup/shared/agent/data/Developing_HumanHeart_Farah/v_fish.h5ad")

In [ ]:
ad